In [1]:
import json
import os
import urllib.error
import urllib.request

# Das Gateway (Traefik) routet /api/v1/board-types an den Board-Registry-Service.
API = os.environ.get("API_BASE_URL", "http://13.63.159.30")
ALICE = {"email": "alice@teamboard.local", "password": os.environ.get("SEED_ALICE_PASSWORD", "AliceSecret123!")}


def call(method, path, body=None, token=None):
    """Minimaler JSON-HTTP-Client. Gibt (status_code, parsed_json) zurück."""
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(f"{API}{path}", data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req) as resp:
            return resp.status, json.loads(resp.read() or "null")
    except urllib.error.HTTPError as e:
        raw = e.read()
        try:
            return e.code, json.loads(raw)
        except Exception:
            return e.code, {"body": raw.decode(errors="replace")}



In [2]:
def register(definition):
    status, resp = call("POST", "/api/v1/board-types", body=definition, token=TOKEN)
    if status == 201:
        print(f"  registriert: {definition['type']} (built_in={resp['data']['built_in']})")
    elif status == 409:
        print(f"  {definition['type']}: existiert bereits (409) — übersprungen")
    else:
        print(f"  FEHLER {status}: {resp}")
    return status, resp


In [4]:
# status, _ = call("POST", "/api/v1/auth/register", ALICE)
# print("register:", status)

status, login = call("POST", "/api/v1/auth/login", ALICE)
assert status == 200, f"Login fehlgeschlagen: {status} {login}"
TOKEN = login["data"]["access_token"]
print("login: ok, Token erhalten")

login: ok, Token erhalten


In [7]:
gantt = {
    "type": "gantt",
    "display_name": "Gantt (Timeline)",
    "icon": "\U0001F4CA",
    "default_columns": [
        {"name": "Planned", "position": 0, "status": "open"},
        {"name": "In Progress", "position": 1, "status": "in_progress"},
        {"name": "Done", "position": 2, "status": "done"},
    ],
    "default_config": {},
    "config_schema": {},
    "presentation": {
        "view": "timeline",
        "view_config": {
            "start_field": "start_date",
            "end_field": "due_date",
            "group_by": "column",
            "color_by": "priority",
        },
        "card": {"fields": ["priority", "due_date"], "color_by": "priority"},
    },
}

register(gantt)

  registriert: gantt (built_in=False)


(201,
 {'data': {'type': 'gantt',
   'display_name': 'Gantt (Timeline)',
   'icon': '📊',
   'default_columns': [{'name': 'Planned', 'position': 0, 'status': 'open'},
    {'name': 'In Progress', 'position': 1, 'status': 'in_progress'},
    {'name': 'Done', 'position': 2, 'status': 'done'}],
   'default_config': {},
   'config_schema': {},
   'presentation': {'card': {'color_by': 'priority',
     'fields': ['priority', 'due_date']},
    'view': 'timeline',
    'view_config': {'color_by': 'priority',
     'end_field': 'due_date',
     'group_by': 'column',
     'start_field': 'start_date'}},
   'built_in': False,
   'created_by': '6b665a99-c6d7-42d9-b66b-ca2c55406748',
   'created_at': '2026-07-13T18:08:12.699732Z',
   'updated_at': '2026-07-13T18:08:12.699732Z'}})

In [8]:
status, _ = call("DELETE", "/api/v1/board-types/gantt", token=TOKEN)
print("delete gantt:", status, "(204 = gelöscht)")

delete gantt: 204 (204 = gelöscht)
